# Aeolus vs MRI-JMA — Williamson Test Case 5 comparison

Runs (or loads) Aeolus Williamson-5 trajectories and compares **physically
identical** fields against the frozen reference package produced by
`mri_w5_reference_preparation.ipynb`.

**Field contract** (audited in `notebooks/W5_MRI_SEMANTIC_AUDIT.md`; the
Aeolus identities are verified from source, `physics/shallow_water.py:9-20`):

```
MRI:     free_surface_height = archived h
         layer_depth         = archived h - topography_height

Aeolus:  layer_depth         = (Phi0 + phi) / gravity
         free_surface_height = (Phi0 + phi + phi_s) / gravity
         topography_height   = phi_s / gravity        (band-limited cone)
```

**Gate:** a 15-day integration is launched only after the day-zero physical
contract passes (or after an explicit user override). `MODE = "verify"`
performs exactly: reference-package verification, field-contract
verification, day-zero comparison, and a smoke run of a few simulated hours
— then stops.

Requires a **GPU runtime** for every mode except `postprocess` (Aeolus is
CuPy-based).


In [ ]:
MODE = "verify"
# allowed:
#   "verify"       - package check + field contract + day-zero comparison +
#                    short smoke run; never starts a 15-day integration
#   "run_t42"      - 15-day T42 (64x128, l_max 42) run, then postprocess
#   "run_t63"      - 15-day T63 (96x192, l_max 63) run, then postprocess
#   "run_both"     - both runs, then postprocess
#   "postprocess"  - figures + metrics from previously saved outputs
#                    (CPU-only; NEVER reruns a simulation)

FORCE_RERUN = False

from pathlib import Path

BASE_DIR = Path("/content") if Path("/content").is_dir() else Path.cwd()

AEOLUS_REPO_URL = "https://github.com/AlexandreEros/Aeolus"
AEOLUS_COMMIT = "580c566a85aa80895163ce3c4abb8a630807bfd2"
AEOLUS_DIR = BASE_DIR / "aeolus-src"          # disposable clone

REFERENCE_DIR = BASE_DIR / "mri-w5-reference"  # Notebook A output
RESULTS_DIR = BASE_DIR / "w5-validation"       # this notebook's outputs

SMOKE_HOURS = 6.0  # smoke-run length in SIMULATED hours (never a full run)

# Explicit user override: launch 15-day runs although the day-zero physical
# contract failed (documented mismatch experiments only -- results are then
# NOT a validation).
ALLOW_RUN_WITH_FAILED_DAY0_CONTRACT = False

# Day-zero physical-contract tolerances. Winds must agree to interpolation
# noise; height fields must agree far below the topography scale (2000 m).
DAY0_TOL = {"wind_max_abs_ms": 0.05,
            "height_max_abs_m": 5.0,
            "height_rel_l2": 5.0e-4}

RUNS = {"t42": {"nlat": 64, "nlon": 128, "lmax": 42},
        "t63": {"nlat": 96, "nlon": 192, "lmax": 63}}
EXPECTED_SCHEMA = "mri-w5-reference/1"

_ALLOWED = {"verify", "run_t42", "run_t63", "run_both", "postprocess"}
assert MODE in _ALLOWED, f"MODE must be one of {sorted(_ALLOWED)}"
NEED_MODEL = MODE in {"verify", "run_t42", "run_t63", "run_both"}
RUN_TARGETS = {"run_t42": ["t42"], "run_t63": ["t63"],
               "run_both": ["t42", "t63"]}.get(MODE, [])
print(f"MODE={MODE}  FORCE_RERUN={FORCE_RERUN}  "
      f"builds Aeolus model (GPU): {NEED_MODEL}")


In [ ]:
# Mount Google Drive ONLY if a configured path actually lives on Drive.
_paths = [REFERENCE_DIR, RESULTS_DIR, AEOLUS_DIR]
if any(str(p).startswith("/content/drive") for p in _paths):
    from google.colab import drive
    drive.mount("/content/drive")
else:
    print("no Google Drive paths configured; Drive not mounted")


In [ ]:
# Verify the frozen reference package (hashes, schema, field contract).
# NumPy only -- runs in every MODE, on CPU.
import hashlib
import json

import numpy as np


def sha256_of(path, chunk=1 << 20):
    digest = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            block = f.read(chunk)
            if not block:
                break
            digest.update(block)
    return digest.hexdigest()


def weighted_metrics(diff, ref, gl_weights):
    """Gauss-Legendre area-weighted error metrics on one (nlat, nlon) field."""
    w = np.broadcast_to(gl_weights[:, None], diff.shape)
    wsum = w.sum()
    denom = float((ref ** 2 * w).sum())
    return {
        "weighted_mae": float((np.abs(diff) * w).sum() / wsum),
        "weighted_rms": float(np.sqrt((diff ** 2 * w).sum() / wsum)),
        "rel_l2": float(np.sqrt((diff ** 2 * w).sum() / denom))
                  if denom > 1e-12 else float("nan"),
        "max_abs": float(np.abs(diff).max()),
        "signed_min": float(diff.min()),
        "signed_max": float(diff.max()),
    }


manifest = json.loads((REFERENCE_DIR / "manifest.json").read_text())
assert manifest["schema_version"] == EXPECTED_SCHEMA, (
    f"reference schema {manifest['schema_version']!r} != {EXPECTED_SCHEMA!r}")

REQUIRED_FIELDS = {"free_surface_height", "layer_depth", "topography_height",
                   "u", "v", "latitude", "longitude", "gl_weights", "time"}

reference = {}
for name, info in manifest["packages"].items():
    path = REFERENCE_DIR / info["file"]
    digest = sha256_of(path)
    assert digest == info["sha256"], (
        f"{path.name}: sha256 {digest} != manifest {info['sha256']}")
    with np.load(path) as z:
        reference[name] = {k: np.asarray(z[k]) for k in z.files}

for name, ref in reference.items():
    missing = REQUIRED_FIELDS - set(ref)
    assert not missing, f"{name}: missing fields {missing}"
    nlat, nlon = RUNS[name]["nlat"], RUNS[name]["nlon"]
    nt = len(manifest["times_days"])
    assert ref["free_surface_height"].shape == (nt, nlat, nlon)
    assert ref["layer_depth"].shape == (nt, nlat, nlon)
    assert ref["topography_height"].shape == (nlat, nlon)
    assert np.allclose(ref["time"], manifest["times_days"])

    # The physical field contract, at every saved time and grid point.
    ident = (ref["free_surface_height"] - ref["topography_height"][None]
             - ref["layer_depth"])
    assert np.abs(ident).max() < 1e-9, (
        f"{name}: layer_depth != free_surface_height - topography_height")

    # Grid convention: exactly the Aeolus gauss-latlon nodes (N -> S).
    x, w = np.polynomial.legendre.leggauss(nlat)
    order = np.argsort(-x)
    lat_expected = np.rad2deg(np.pi / 2.0 - np.arccos(x[order]))
    assert np.allclose(ref["latitude"], lat_expected, atol=1e-10), (
        f"{name}: latitudes are not the leggauss nodes in N->S order")
    assert np.allclose(ref["longitude"], np.arange(nlon) * (360.0 / nlon))
    assert np.allclose(ref["gl_weights"], w[order])

    assert (ref["layer_depth"] > 0.0).all()
    assert np.isfinite(ref["free_surface_height"]).all()
    print(f"{name}: reference package verified "
          "(hash, schema, identity, grid convention)")

REFERENCE_OK = True


In [ ]:
# Clone the exact Aeolus commit and install ONLY missing dependencies.
import importlib
import os
import subprocess
import sys

if NEED_MODEL:
    if not (AEOLUS_DIR / ".git").exists():
        subprocess.check_call(["git", "clone", AEOLUS_REPO_URL,
                               str(AEOLUS_DIR)])
    subprocess.check_call(["git", "-C", str(AEOLUS_DIR), "fetch", "origin"])
    subprocess.check_call(["git", "-C", str(AEOLUS_DIR), "checkout",
                           AEOLUS_COMMIT])
    head = subprocess.check_output(
        ["git", "-C", str(AEOLUS_DIR), "rev-parse", "HEAD"],
        text=True).strip()
    assert head == AEOLUS_COMMIT, f"checkout produced {head}"
    if str(AEOLUS_DIR / "src") not in sys.path:
        sys.path.insert(0, str(AEOLUS_DIR / "src"))

    # Colab already ships numpy/scipy/matplotlib. Aeolus additionally needs
    # CuPy. Do NOT install the repository's requirements.txt: it pins a
    # CUDA-11 CuPy wheel and exact numpy/scipy versions that fight Colab.
    try:
        importlib.import_module("cupy")
        installed_now = False
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install",
                               "cupy-cuda12x"])
        installed_now = True
        importlib.invalidate_caches()
        try:
            importlib.import_module("cupy")
        except Exception:
            print("=" * 72)
            print("CuPy was installed but cannot be loaded into this live "
                  "kernel.")
            print("Restart the runtime (Runtime > Restart session) and "
                  "re-run this")
            print("notebook from the top. Nothing else needs to change.")
            print("=" * 72)
            raise SystemExit(0)

    import cupy
    try:
        n_gpu = cupy.cuda.runtime.getDeviceCount()
    except Exception:
        n_gpu = 0
    if n_gpu == 0:
        raise RuntimeError(
            "Aeolus is CuPy-based and needs a GPU runtime for this MODE. "
            "Switch to a GPU runtime, or use MODE='postprocess'.")
    print(f"Aeolus ready at {AEOLUS_DIR} (commit {head[:12]}), "
          f"CuPy sees {n_gpu} GPU(s)"
          + (" [cupy installed this session]" if installed_now else ""))
else:
    print(f"MODE={MODE} does not build the model; skipping clone and deps")


In [ ]:
# Model construction + physical-field extraction (the audited formulas).
if NEED_MODEL:
    import cupy as cp
    from planetary_sandbox.physics.shallow_water import (ShallowWaterModel,
                                                         ShallowWaterState)
    from planetary_sandbox.physics.topography import Topography
    from planetary_sandbox.planet import Planet, PlanetaryParameters
    from planetary_sandbox.run.swe.config import (W5_DAY_HOURS, W5_GRAVITY,
                                                  W5_MEAN_DEPTH_M,
                                                  W5_RADIUS_M)
    from planetary_sandbox.run.swe.diagnostics import potential_enstrophy
    from planetary_sandbox.run.swe.initial_conditions import make_swe_ic

    def build_w5_model(nlat, nlon, lmax):
        """Exactly `aeolus run swe --scenario williamson5 --backend
        gauss-latlon` (cli/swe.py:_execute_solver / _w5_planet_params)."""
        params = PlanetaryParameters.ideal_sphere(
            radius_m=W5_RADIUS_M, sidereal_day_s=W5_DAY_HOURS * 3600.0)
        planet = Planet.generate(params=params, grid_resolution=4,
                                 l_max=lmax, product_quadrature="fine",
                                 grid_type="latlon", nlat=nlat, nlon=nlon)
        topography = Topography.williamson5_cone(planet)
        return ShallowWaterModel(planet, gravity=W5_GRAVITY,
                                 mean_depth=W5_MEAN_DEPTH_M,
                                 topography=topography)

    def extract_fields(model, coeffs, nlat, nlon):
        """Physical fields of one spectral SWE state:

            aeolus_layer_depth         = (Phi0 + phi) / g
            aeolus_free_surface_height = (Phi0 + phi + phi_s) / g
            aeolus_topography_height   = phi_s / g       (band-limited cone)
        """
        state = ShallowWaterState(cp.asarray(coeffs))
        gravity = model.gravity
        phi = model.sh.inv_transform(state.coeffs[2]).real
        phi_s = model.surface_geopotential_on_state_grid()
        assert phi_s is not None, "W5 model must carry topography"
        layer_depth = (model.phi0 + phi) / gravity
        free_surface = layer_depth + phi_s / gravity
        u, v = model.wind_on_state_grid(state)
        out = {"layer_depth": layer_depth,
               "free_surface_height": free_surface,
               "topography_height": phi_s / gravity, "u": u, "v": v}
        return {k: cp.asnumpy(val).reshape(nlat, nlon).astype(np.float64)
                for k, val in out.items()}

    print("model helpers defined")


In [ ]:
# Day-zero inspection and physical-contract comparison (BEFORE any long
# integration). Grid/orientation problems are reported separately from
# physical initial-condition mismatch.
DAY0_CONTRACT = {}
CONTRACT_PASSED = None

if NEED_MODEL:
    RESULTS_DIR.mkdir(parents=True, exist_ok=True)
    day0_report = {}
    targets_day0 = RUN_TARGETS if RUN_TARGETS else list(RUNS)

    for name in targets_day0:
        spec = RUNS[name]
        ref = reference[name]
        model = build_w5_model(spec["nlat"], spec["nlon"], spec["lmax"])
        state0 = make_swe_ic("williamson5", model)
        f0 = extract_fields(model, cp.asnumpy(state0.coeffs),
                            spec["nlat"], spec["nlon"])

        # --- grid / orientation / unit checks (NOT physics) ---------------
        lat_model = np.rad2deg(cp.asnumpy(model.grid.latitudes))
        lon_model = np.rad2deg(cp.asnumpy(model.grid.longitudes))
        assert np.allclose(lat_model, ref["latitude"], atol=1e-9), (
            f"{name}: GRID mismatch (latitude nodes/ordering)")
        assert np.allclose(lon_model, ref["longitude"], atol=1e-9), (
            f"{name}: GRID mismatch (longitude nodes)")

        # --- physical comparison ------------------------------------------
        gl_w = ref["gl_weights"]
        rows = {}
        for field in ("free_surface_height", "u", "v"):
            rows[field] = weighted_metrics(f0[field] - ref[field][0],
                                           ref[field][0], gl_w)

        # Layer depth: Aeolus carries the BAND-LIMITED cone while the
        # reference carries the analytic cone, so their depths differ by a
        # static, cone-local terrain-representation term (audit section 4)
        # even for identical physical initial conditions. Report the raw
        # difference, then remove exactly that term and hold the remainder
        # to the strict tolerance.
        d_depth = f0["layer_depth"] - ref["layer_depth"][0]
        terrain_repr = (ref["topography_height"]
                        - f0["topography_height"])  # analytic - band-limited
        rows["layer_depth"] = weighted_metrics(d_depth,
                                               ref["layer_depth"][0], gl_w)
        rows["layer_depth_minus_terrain_repr"] = weighted_metrics(
            d_depth - terrain_repr, ref["layer_depth"][0], gl_w)

        # Structural diagnosis: does the free-surface difference EQUAL the
        # topography? (signature of the thickness-vs-free-surface
        # convention mismatch found by the semantic audit)
        d_fs = f0["free_surface_height"] - ref["free_surface_height"][0]
        rows["fs_diff_minus_topography"] = weighted_metrics(
            d_fs - ref["topography_height"],
            ref["free_surface_height"][0], gl_w)

        winds_ok = (rows["u"]["max_abs"] < DAY0_TOL["wind_max_abs_ms"]
                    and rows["v"]["max_abs"] < DAY0_TOL["wind_max_abs_ms"])
        heights_ok = all(
            rows[f]["max_abs"] < DAY0_TOL["height_max_abs_m"]
            and rows[f]["rel_l2"] < DAY0_TOL["height_rel_l2"]
            for f in ("free_surface_height",
                      "layer_depth_minus_terrain_repr"))
        DAY0_CONTRACT[name] = bool(winds_ok and heights_ok)
        day0_report[name] = {"metrics": rows, "winds_ok": winds_ok,
                             "heights_ok": heights_ok,
                             "contract_passed": DAY0_CONTRACT[name]}

        print(f"\n=== day-zero physical contract: {name} "
              f"(l_max={spec['lmax']}, {spec['nlat']}x{spec['nlon']}) ===")
        for field, m in rows.items():
            print(f"  {field:26s} max|d| = {m['max_abs']:12.6f}   "
                  f"wRMS = {m['weighted_rms']:12.6f}   "
                  f"relL2 = {m['rel_l2']:.3e}")
        print(f"  winds match: {winds_ok} | height fields match: "
              f"{heights_ok}")
        print("  (layer_depth is judged after removing the static "
              "analytic-vs-band-limited terrain-representation term; the "
              "raw layer_depth row above includes it and is bounded by "
              "the cone projection residual.)")
        if winds_ok and not heights_ok:
            frac = (rows["fs_diff_minus_topography"]["max_abs"]
                    / max(rows["free_surface_height"]["max_abs"], 1e-30))
            if frac < 0.15:
                print("  DIAGNOSIS: INITIAL-CONDITION MISMATCH (not grid/"
                      "unit/orientation:")
                print("  winds and grids agree). The Aeolus free surface "
                      "equals the MRI free")
                print("  surface PLUS the topography. Aeolus prescribes the "
                      "case-2 field as")
                print("  layer THICKNESS; MRI/Williamson prescribe it as "
                      "the FREE SURFACE.")
                print("  Physically different initial-value problems. See "
                      "notebooks/W5_MRI_SEMANTIC_AUDIT.md.")
        print(f"  CONTRACT {'PASSED' if DAY0_CONTRACT[name] else 'FAILED'}")

        # Release this grid's model before building the next one (the
        # spectral matrices are large; keeps peak GPU memory to one model).
        del state0, model
        cp.get_default_memory_pool().free_all_blocks()

    CONTRACT_PASSED = all(DAY0_CONTRACT.values())
    (RESULTS_DIR / "day0_contract.json").write_text(
        json.dumps({"aeolus_commit": AEOLUS_COMMIT,
                    "tolerances": DAY0_TOL, "report": day0_report,
                    "contract_passed": CONTRACT_PASSED}, indent=2))
    print(f"\nday-zero contract passed on all inspected grids: "
          f"{CONTRACT_PASSED}")
    print("report saved to", RESULTS_DIR / "day0_contract.json")


In [ ]:
# Smoke run: a few SIMULATED hours through the real CLI, to prove the
# integration machinery works. This is an infrastructure check, not science;
# it runs regardless of the contract verdict and is never a full run.
if MODE == "verify":
    smoke_dir = RESULTS_DIR / "smoke"
    smoke_dir.mkdir(parents=True, exist_ok=True)
    env = dict(os.environ)
    env["PYTHONPATH"] = str(AEOLUS_DIR / "src")
    cmd = [sys.executable, "-m", "planetary_sandbox.cli.main", "run", "swe",
           "--scenario", "williamson5", "--backend", "gauss-latlon",
           "--nlat", "64", "--nlon", "128", "--l-max", "42",
           "--days", f"{SMOKE_HOURS / 24.0}", "--n-snapshots", "2",
           "--no-plots", "--out", str(smoke_dir), "--experiment", "smoke"]
    print(" ".join(cmd))
    subprocess.check_call(cmd, env=env)

    run_dirs = sorted((smoke_dir / "smoke").glob("*williamson5*"))
    assert run_dirs, "smoke run produced no run directory"
    coeffs = np.load(run_dirs[-1] / "swe_coeffs.npy")
    times = np.load(run_dirs[-1] / "swe_snapshot_times.npy")
    assert np.isfinite(coeffs).all(), "smoke run produced non-finite state"
    assert abs(times[-1] - SMOKE_HOURS * 3600.0) < 1.0
    print(f"\nsmoke run OK: {coeffs.shape[0]} snapshots over "
          f"{SMOKE_HOURS} simulated hours, all coefficients finite")
    print("run directory:", run_dirs[-1])
    print("\nMODE='verify' is complete. Review the day-zero contract above "
          "before launching any 15-day run.")


In [ ]:
# 15-day integrations (T42 / T63) -- gated on the day-zero contract.
# Valid completed outputs are reused unless FORCE_RERUN=True.
import datetime
import shutil

if RUN_TARGETS:
    if not CONTRACT_PASSED and not ALLOW_RUN_WITH_FAILED_DAY0_CONTRACT:
        print("=" * 72)
        print("REFUSING to launch 15-day runs: the day-zero physical "
              "contract FAILED.")
        print("Aeolus and MRI currently integrate DIFFERENT initial-value "
              "problems")
        print("(see the day-zero cell and "
              "notebooks/W5_MRI_SEMANTIC_AUDIT.md).")
        print("Correct the Aeolus initial condition on a separate branch, "
              "or set")
        print("ALLOW_RUN_WITH_FAILED_DAY0_CONTRACT = True for an explicit,")
        print("clearly-labeled mismatch experiment.")
        print("=" * 72)
        RUN_TARGETS = []

for name in RUN_TARGETS:
    spec = RUNS[name]
    done_file = RESULTS_DIR / f"aeolus_{name}_COMPLETE.json"
    out_npz = RESULTS_DIR / f"aeolus_w5_{name}.npz"

    if done_file.exists() and out_npz.exists() and not FORCE_RERUN:
        info = json.loads(done_file.read_text())
        if sha256_of(out_npz) == info.get("npz_sha256"):
            print(f"{name}: valid completed output found -- reusing "
                  "(FORCE_RERUN=False)")
            continue
        print(f"{name}: completion record does not match outputs; rerunning")

    run_out = RESULTS_DIR / "runs" / name
    run_out.mkdir(parents=True, exist_ok=True)
    env = dict(os.environ)
    env["PYTHONPATH"] = str(AEOLUS_DIR / "src")
    cmd = [sys.executable, "-m", "planetary_sandbox.cli.main", "run", "swe",
           "--scenario", "williamson5", "--backend", "gauss-latlon",
           "--nlat", str(spec["nlat"]), "--nlon", str(spec["nlon"]),
           "--l-max", str(spec["lmax"]), "--days", "15",
           "--n-snapshots", "4", "--no-plots",
           "--out", str(run_out), "--experiment", "w5-mri"]
    print(" ".join(cmd))
    subprocess.check_call(cmd, env=env)

    run_dir = sorted((run_out / "w5-mri").glob("*williamson5*"))[-1]
    coeffs = np.load(run_dir / "swe_coeffs.npy")
    times_s = np.load(run_dir / "swe_snapshot_times.npy")
    assert coeffs.shape[0] == 4 and np.isfinite(coeffs).all()

    # Extract plain physical fields for every snapshot (GPU, once, here --
    # postprocessing later never needs CuPy).
    model = build_w5_model(spec["nlat"], spec["nlon"], spec["lmax"])
    stacks = {k: [] for k in ("free_surface_height", "layer_depth",
                              "u", "v")}
    enstrophy = []
    for k in range(coeffs.shape[0]):
        fk = extract_fields(model, coeffs[k], spec["nlat"], spec["nlon"])
        for key in stacks:
            stacks[key].append(fk[key])
        enstrophy.append(potential_enstrophy(
            model, ShallowWaterState(cp.asarray(coeffs[k]))))
    topo_bl = fk["topography_height"]  # static

    np.savez_compressed(
        out_npz,
        **{k: np.stack(v) for k, v in stacks.items()},
        topography_height_bandlimited=topo_bl,
        latitude=np.rad2deg(cp.asnumpy(model.grid.latitudes)),
        longitude=np.rad2deg(cp.asnumpy(model.grid.longitudes)),
        gl_weights=reference[name]["gl_weights"],
        time=times_s / 86400.0,
        potential_enstrophy=np.asarray(enstrophy))

    del model
    cp.get_default_memory_pool().free_all_blocks()

    csv_src = run_dir / "diagnostics" / "timeseries.csv"
    if csv_src.exists():
        shutil.copy(csv_src, RESULTS_DIR / f"aeolus_{name}_timeseries.csv")

    done_file.write_text(json.dumps({
        "aeolus_commit": AEOLUS_COMMIT,
        "run_dir": str(run_dir),
        "npz_sha256": sha256_of(out_npz),
        "contract_passed_at_launch": bool(CONTRACT_PASSED),
        "explicit_override_used": bool(not CONTRACT_PASSED),
        "created_utc": datetime.datetime.now(
            datetime.timezone.utc).isoformat()}, indent=2))
    print(f"{name}: run complete; fields exported to {out_npz.name}")


In [ ]:
# Postprocess: comparisons at days 0/5/10/15 + metrics + conservation.
# CPU/NumPy/Matplotlib only. NEVER runs or reruns a simulation.
import csv as _csv

import matplotlib.pyplot as plt

if MODE == "postprocess" or RUN_TARGETS:
    FIG_DIR = RESULTS_DIR / "figures"
    FIG_DIR.mkdir(parents=True, exist_ok=True)

    available = []
    for name in RUNS:
        done_file = RESULTS_DIR / f"aeolus_{name}_COMPLETE.json"
        out_npz = RESULTS_DIR / f"aeolus_w5_{name}.npz"
        if done_file.exists() and out_npz.exists():
            info = json.loads(done_file.read_text())
            assert sha256_of(out_npz) == info["npz_sha256"], (
                f"{name}: saved fields do not match the completion record")
            available.append(name)
    if not available:
        print("no completed Aeolus outputs under", RESULTS_DIR,
              "- run MODE='run_t42'/'run_t63' first "
              "(postprocess never launches simulations)")

    metrics_rows = []
    for name in available:
        z = dict(np.load(RESULTS_DIR / f"aeolus_w5_{name}.npz"))
        ref = reference[name]
        info = json.loads(
            (RESULTS_DIR / f"aeolus_{name}_COMPLETE.json").read_text())
        assert np.allclose(z["time"], ref["time"]), (
            f"{name}: snapshot days differ from reference days")
        assert np.allclose(z["latitude"], ref["latitude"], atol=1e-9)
        gl_w = ref["gl_weights"]
        if info.get("explicit_override_used"):
            print(f"NOTE: {name} was integrated with a FAILED day-zero "
                  "contract (explicit override); the comparison below "
                  "measures a KNOWN initial-condition mismatch, not Aeolus "
                  "trajectory error.")

        for k, day in enumerate(ref["time"]):
            for field in ("free_surface_height", "layer_depth", "u", "v"):
                m = weighted_metrics(z[field][k] - ref[field][k],
                                     ref[field][k], gl_w)
                metrics_rows.append({"grid": name, "day": float(day),
                                     "field": field, **m})

        # comparison maps: MRI | Aeolus | signed error
        lat_t, lon_t = ref["latitude"], ref["longitude"]
        for k, day in enumerate(ref["time"]):
            speed_ref = np.hypot(ref["u"][k], ref["v"][k])
            speed_aeo = np.hypot(z["u"][k], z["v"][k])
            rows_spec = [
                ("free_surface_height", ref["free_surface_height"][k],
                 z["free_surface_height"][k]),
                ("layer_depth", ref["layer_depth"][k], z["layer_depth"][k]),
                ("wind speed", speed_ref, speed_aeo),
            ]
            fig, axes = plt.subplots(3, 3, figsize=(15, 9),
                                     constrained_layout=True)
            for r, (label, fr, fa) in enumerate(rows_spec):
                err = fa - fr
                vmax = float(np.abs(err).max()) or 1.0
                for c, (field, cmap, norm) in enumerate([
                        (fr, "viridis", None), (fa, "viridis", None),
                        (err, "RdBu_r", vmax)]):
                    ax = axes[r, c]
                    if norm is None:
                        im = ax.pcolormesh(lon_t, lat_t, field,
                                           shading="nearest", cmap=cmap)
                    else:
                        im = ax.pcolormesh(lon_t, lat_t, field,
                                           shading="nearest", cmap=cmap,
                                           vmin=-norm, vmax=norm)
                    fig.colorbar(im, ax=ax, shrink=0.85)
                    ax.set_title([f"MRI {label}", f"Aeolus {label}",
                                  f"signed error (Aeolus - MRI)"][c])
            fig.suptitle(f"{name} - day {int(day)}")
            fig.savefig(FIG_DIR / f"compare_{name}_day{int(day):02d}.png",
                        dpi=110)
            plt.show()
            plt.close(fig)

        # conservation: energy drift from the per-step CSV, potential-
        # enstrophy drift from the exported snapshots
        csv_path = RESULTS_DIR / f"aeolus_{name}_timeseries.csv"
        if csv_path.exists():
            with open(csv_path, newline="") as f:
                rows = list(_csv.DictReader(f))
            energy = np.asarray([float(r["total_energy"]) for r in rows])
            mass = np.asarray([float(r["total_mass"]) for r in rows])
            print(f"{name}: energy drift (E_end-E_0)/E_0 = "
                  f"{(energy[-1] - energy[0]) / energy[0]:+.3e} ; "
                  f"mass drift = {(mass[-1] - mass[0]) / mass[0]:+.3e}")
        ens = z["potential_enstrophy"]
        print(f"{name}: potential-enstrophy drift (Z_end-Z_0)/Z_0 = "
              f"{(ens[-1] - ens[0]) / ens[0]:+.3e}")

    if metrics_rows:
        out_csv = RESULTS_DIR / "comparison_metrics.csv"
        with open(out_csv, "w", newline="") as f:
            writer = _csv.DictWriter(f, fieldnames=list(metrics_rows[0]))
            writer.writeheader()
            writer.writerows(metrics_rows)
        print("\nmetrics written to", out_csv)
        hdr = f"{'grid':5s} {'day':>4s} {'field':22s} {'wMAE':>12s} " \
              f"{'wRMS':>12s} {'relL2':>10s} {'max|d|':>12s}"
        print(hdr)
        for r in metrics_rows:
            print(f"{r['grid']:5s} {r['day']:4.0f} {r['field']:22s} "
                  f"{r['weighted_mae']:12.4e} {r['weighted_rms']:12.4e} "
                  f"{r['rel_l2']:10.3e} {r['max_abs']:12.4e}")

    if len(available) == 2:
        print("\nresolution dependence (relL2 of free_surface_height, "
              "T42 vs T63):")
        for day in reference["t42"]["time"]:
            vals = {r["grid"]: r["rel_l2"] for r in metrics_rows
                    if r["field"] == "free_surface_height"
                    and r["day"] == day}
            print(f"  day {day:4.0f}:  t42 {vals.get('t42', float('nan')):.3e}"
                  f"   t63 {vals.get('t63', float('nan')):.3e}")


## Reading the results — what a difference means

| signature | classification |
|---|---|
| day-0 height differences ≈ the topography field, winds agree | **field-definition / initial-condition mismatch** (thickness vs free-surface convention — the current state; see the audit) |
| day-0 differences smooth and O(interpolation error), winds agree | contract passed; later differences are model error |
| day-0 wind or grid assertions fail | grid / orientation / unit problem in the pipeline (fix before interpreting anything) |
| differences grow with time from a passing day-0 | **numerical trajectory error** (the thing being validated) |
| T63 errors < T42 errors against the same reference | **resolution dependence** (expected convergence) |
| energy / potential-enstrophy drift | **conservation behavior** of the Aeolus integration itself (independent of MRI) |

Static cone-shaped residuals of a few tens of metres in `layer_depth` near
the mountain (with a matching free surface) are the **band-limited
topography representation** of Aeolus vs the analytic cone — report them
separately from trajectory error.
